# Session 1: a number without a unit is not an answer
**A. Real-world problem.** A friend texts "the car did 60". Sixty what? Metres per second,
kilometres per hour, miles per hour? Those are very different speeds. Today you build a unit
converter, learn the SI prefixes, and see why every number must carry its unit.

**Plan.** (1) tables of units with their size in SI, (2) a converter, (3) prefixes,
(4) significant figures, (5) a picture of what one bare number can mean.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: every unit is "this many SI units"
SI = the international system of units. Its base units for length, mass and time are the
metre (m), kilogram (kg) and second (s). Every other unit is defined as a fixed multiple:
1 km = 1000 m, 1 ft = 0.3048 m (exactly), 1 mile = 1609.344 m (exactly).
We store the multiplier and convert through the SI unit.

In [ ]:
LENGTH = {"m": 1.0, "km": 1000.0, "cm": 0.01, "mm": 0.001, "in": 0.0254, "ft": 0.3048,
          "mi": 1609.344, "nmi": 1852.0}          # metres per unit
SPEED = {"m/s": 1.0, "km/h": 1000.0 / 3600.0, "mph": 1609.344 / 3600.0,
         "knot": 1852.0 / 3600.0, "ft/s": 0.3048}  # (metres per second) per unit


def convert(value, from_unit, to_unit, table):
    """Convert a number between two units of the same kind. Raises KeyError for unknown units."""
    return value * table[from_unit] / table[to_unit]

## Step 2: prefixes are just powers of ten
kilo = 10^3, mega = 10^6, milli = 10^-3, micro = 10^-6 ... A prefix is a shorthand
multiplier, so 5 km = 5 * 10^3 m = 5000 m.

In [ ]:
PREFIX = {"G": 9, "M": 6, "k": 3, "": 0, "m": -3, "u": -6, "n": -9}   # symbol -> power of ten


def with_prefix(value, symbol):
    """Value in the base unit for `value` written with a prefix (u means micro)."""
    return value * 10.0 ** PREFIX[symbol]


def best_prefix(value):
    """Choose the prefix that puts the number between 1 and 1000. Returns (number, symbol)."""
    if value == 0:
        return 0.0, ""
    power = 3 * int(np.floor(np.log10(abs(value)) / 3))
    power = max(-9, min(9, power))
    symbol = [s for s, p in PREFIX.items() if p == power][0]
    return value / 10.0 ** power, symbol

## Step 3: significant figures say how much to trust a number
A measurement of 12.3 m is good to about 0.1 m. Writing 12.3456 m would claim more than
you know. `round_sig(x, n)` keeps n significant figures.

In [ ]:
def round_sig(x, n):
    if x == 0:
        return 0.0
    return float(round(x, n - 1 - int(np.floor(np.log10(abs(x))))))


def label(value, unit, n=3):
    """A number is only an answer with its unit attached."""
    return f"{round_sig(value, n):g} {unit}"

## Step 4: see the problem
The same bare number 60 means five different speeds, depending on the unit.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    names = list(SPEED)
    ms = [60 * SPEED[u] for u in names]
    for u, v in zip(names, ms):
        print(f"60 {u:5s} = {v:7.3f} m/s")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(names, ms, color="#2a9d8f")
    ax.set_xlabel("unit written after the number 60")
    ax.set_ylabel("speed in SI (m/s)")
    ax.set_title("The same number 60 means five different speeds")
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "same_number.png", dpi=120)

    n = np.linspace(0, 100, 101)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for u in names:
        ax2.plot(n, n * SPEED[u], label=u)
    ax2.set_xlabel("number you read (no unit)")
    ax2.set_ylabel("speed (m/s)")
    ax2.set_title("Each unit is a straight line through the origin; its slope is the unit size")
    ax2.grid(alpha=0.3)
    ax2.legend(title="unit")
    fig2.tight_layout()
    fig2.savefig(OUT / "unit_lines.png", dpi=120)
    print("72 km/h =", label(convert(72, "km/h", "m/s", SPEED), "m/s"))
    print("60 mph  =", label(convert(60, "mph", "km/h", SPEED), "km/h"))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the numbers in `lesson.py` to convert 5 knots to km/h. Predict the answer before you run it.
2. **Stretch.** Write `to_si(value, unit)` that returns the value in SI for any speed unit and test it on all five units.
3. **Explain.** Explain why the lines in the graph all start at zero and why a unit with a bigger size has a steeper line.

## Assessment
- **A. Conceptual.** Why is '60' not a complete answer to 'how fast was the car?' What must you add, and why does the SI unit help when two people disagree?
- **B. Numerical.** Convert 100 km/h to m/s and to mph. Check that your answers are sensible.
- **C. Graph and simulation interpretation.** In the graph of speed against the number read, which unit gives the highest speed for the number 60, and which the lowest? What does the slope of each line mean?
- **D. Design and optimisation challenge.** Design: a road sign for visitors must show a 50 km/h limit and a mph equivalent. Choose a rounding and justify it, noting the trade-off between clarity and accuracy.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 72 km/h is 20 m/s",
                   abs(L.convert(72, "km/h", "m/s", L.SPEED) - 20.0) < 1e-9))
    checks.append(("worked example: 60 mph is 96.56 km/h (96.6 to 3 figures)",
                   abs(L.convert(60, "mph", "km/h", L.SPEED) - 96.56064) < 1e-4
                   and abs(L.round_sig(L.convert(60, "mph", "km/h", L.SPEED), 3) - 96.6) < 1e-9))
    checks.append(("hand calculation: 5 km is 5000 m", abs(L.with_prefix(5, "k") - 5000.0) < 1e-9))
    checks.append(("1 mile is 1.609344 km exactly", abs(L.convert(1, "mi", "km", L.LENGTH) - 1.609344) < 1e-12))
    checks.append(("there and back again restores the number",
                   abs(L.convert(L.convert(37.5, "ft", "m", L.LENGTH), "m", "ft", L.LENGTH) - 37.5) < 1e-9))
    checks.append(("micro is 10^-6: 250 u is 0.00025", abs(L.with_prefix(250, "u") - 0.00025) < 1e-15))
    num, sym = L.best_prefix(0.0042)
    checks.append(("best prefix of 0.0042 is 4.2 m(illi)", abs(num - 4.2) < 1e-9 and sym == "m"))
    checks.append(("edge: zero converts to zero", L.convert(0.0, "mph", "knot", L.SPEED) == 0.0))
    checks.append(("round_sig(0.012345, 3) is 0.0123", abs(L.round_sig(0.012345, 3) - 0.0123) < 1e-12))
    checks.append(("the bare number 60 is five different speeds",
                   len({round(60 * v, 6) for v in L.SPEED.values()}) == 5))
    try:
        L.convert(1.0, "furlong", "m", L.LENGTH)
        unknown_ok = False
    except KeyError:
        unknown_ok = True
    checks.append(("an unknown unit raises an error instead of guessing", unknown_ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
